In [46]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

In [47]:
np.random.seed(42)
n_samples = 500

# Spam emails
spam = pd.DataFrame({
    'offer_freq': np.random.poisson(3, n_samples//2),
    'free_freq': np.random.poisson(4, n_samples//2),
    'money_freq': np.random.poisson(2, n_samples//2),
    'click_freq': np.random.poisson(3, n_samples//2),
    'num_links': np.random.randint(1, 10, n_samples//2),
    'is_caps': np.random.uniform(0.4, 1.0, n_samples//2),
    'y': 1
})

# Non-spam emails
not_spam = pd.DataFrame({
    'offer_freq': np.random.poisson(0.5, n_samples//2),
    'free_freq': np.random.poisson(0.3, n_samples//2),
    'money_freq': np.random.poisson(0.2, n_samples//2),
    'click_freq': np.random.poisson(0.4, n_samples//2),
    'num_links': np.random.randint(0, 3, n_samples//2),
    'is_caps': np.random.uniform(0.0, 0.4, n_samples//2),
    'y': 0
})

df = pd.concat([spam, not_spam], ignore_index=True)
df = df.sample(frac=1).reset_index(drop=True)  # Shuffle


In [48]:
df.head()

,offer_freq,free_freq,money_freq,click_freq,num_links,is_caps,y
0,0,0,0,0,1,0.374992,0
1,2,1,3,1,3,0.466725,1
2,6,3,2,1,9,0.923593,1
3,1,0,1,1,1,0.118886,0
4,6,3,2,1,6,0.791784,1


In [49]:
x = np.array([df["offer_freq"],df["free_freq"],df["money_freq"],df["click_freq"],df["num_links"],df["is_caps"]]).T
y = np.array(df["y"])

In [50]:
y = y.reshape(-1, 1)


In [51]:
def sigmoid(z):
    z = np.clip(z,-500,500)
    return 1/(1+np.exp(-z))

In [52]:
def loss(x,y,theta,lam):
    m = len(y)
    hx = sigmoid(np.dot(x,theta))
    epsilon = 1e-9
    hx = np.clip(hx,epsilon,1-epsilon)
    l = -(y*np.log(hx)+(1-y)*np.log(1-hx))
    reg = (lam*np.sum(theta[1:]**2))/(2)
    return (np.sum(l)+reg)/m

In [53]:
def grad_decent(x,y,theta,lam,alpha,iter):
    m = len(y)
    theta_tilda =theta.copy()
    theta_tilda[0] = 0
    cost = []

    for i in range(iter):
        hx = sigmoid(np.dot(x,theta))
        error = hx - y

        grad = np.dot(x.T,error) + lam*theta_tilda/m

        theta -= alpha*grad
        cost.append(loss(x, y, theta, lam))

    
    return theta,cost


In [54]:
def predict(theta,x):
    return (sigmoid(np.dot(x,theta))>0.5).flatten()

In [55]:
def logistic_regression(x,y,lam = 1,power = 2,alpha = 0.01,iters = 100 ):
    poly = PolynomialFeatures(degree=2, include_bias=True)
    x_poly = poly.fit_transform(x)
    print(x_poly.shape)
    theta = np.zeros((x_poly.shape[1],1),dtype=np.float64)
    theta, costs = grad_decent(x_poly, y, theta, lam, alpha, iters)
    predicted = predict(theta, x_poly)
    return predicted,theta,costs

In [56]:
power , num = 2,20000
predicted,theta,costs = logistic_regression(x,y,lam=1,power=power,alpha=0.6,iters=num)

(500, 28)


In [57]:
print('The accuracy is {:.2f}%'.format(sum(predicted==y.flatten())/len(y)*100))

The accuracy is 100.00%


In [58]:
theta

array([[-1244.33533898],
       [ -440.27761935],
       [ -541.75097365],
       [ -228.39489756],
       [ -422.71599947],
       [-1231.57492776],
       [  -90.23642538],
       [ -135.2927399 ],
       [  301.47636027],
       [  379.134392  ],
       [  388.80726017],
       [  223.40227437],
       [  187.53419759],
       [  302.16760408],
       [   58.25721044],
       [  377.39638583],
       [  352.17475199],
       [  176.34931702],
       [  -69.73267524],
       [  500.59732229],
       [  298.14941893],
       [  211.50618853],
       [ -301.98491883],
       [  319.44860242],
       [  110.73641487],
       [   40.28485565],
       [  150.79924452],
       [   56.95346982]])